In [1]:
import os
import sys

# Print Python version and its binary path
print("Python version:", sys.version)
print("Python binary path:", sys.executable)

# Print the current working directory
print("Current working directory:", os.getcwd())

# Print sys.path
print("sys.path:", sys.path)


Python version: 3.10.15 (main, Oct  3 2024, 07:27:34) [GCC 11.2.0]
Python binary path: /home/phinguyen/miniconda3/envs/tiki/bin/python
Current working directory: /home/phinguyen/projects/tiki_book_engineering/airflow/dags/pipelines/etl
sys.path: ['/home/phinguyen/miniconda3/envs/tiki/lib/python310.zip', '/home/phinguyen/miniconda3/envs/tiki/lib/python3.10', '/home/phinguyen/miniconda3/envs/tiki/lib/python3.10/lib-dynload', '', '/home/phinguyen/miniconda3/envs/tiki/lib/python3.10/site-packages']


In [2]:
import pandas as pd

from extract import query_data_from_sqliteDB

In [3]:
query = """SELECT * FROM books"""
books_df = query_data_from_sqliteDB(query)

Successfully fetch data from sqliteDB!


In [4]:
# Set display options to show all columns
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)  # Prevent line wrapping

books_df.head(1)

,id,tiki_id,master_id,sku,name,type,short_description,price,list_price,original_price,is_authentic,is_freeship_xtra,is_hero,is_top_brand,return_reason,discount,discount_rate,rating_average,review_count,favourite_count,has_ebook,inventory_status,inventory_type,productset_group_name,data_version,day_ago_created,all_time_quantity_sold,authors,current_seller,categories,specifications,breadcrumbs,is_seller_in_chat_whitelist,is_tier_pricing_available,is_tier_pricing_eligible,crawl_time
0,1,42230121,42230121,3098193178849,Sách Hiểu Về Trái Tim (Tái Bản 2019) - Minh Niệm,simple,Hiểu Về Trái Tim – Cuốn Sách Mở Cửa Thề Giới C...,135000,158000,158000,1,0,1,0,any_reason,23000,15,4.8,3698,0,0,available,instock,,3300,1809,36364,"[{""id"": 5139, ""name"": ""Minh Ni\u1ec7m"", ""slug""...","{""id"": 1, ""sku"": ""8105872042430"", ""name"": ""Tik...","{""id"": 316, ""name"": ""S\u00e1ch ti\u1ebfng Vi\u...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-10-31 21:30:00


In [5]:
# Convert crawl_time to datetime format
books_df['crawl_time'] = pd.to_datetime(books_df['crawl_time'])

# Sort by tiki_id and crawl_time to ensure correct order for diff
books_df = books_df.sort_values(by=['tiki_id', 'crawl_time'])

# Calculate quantity_by_day as the difference in all_time_quantity_sold for each tiki_id
books_df['quantity_by_day'] = books_df.groupby('tiki_id')['all_time_quantity_sold'].diff().fillna('null')

# Display the result
print(books_df)

      id    tiki_id  master_id            sku                                               name          type                                  short_description   price  list_price  original_price  is_authentic  is_freeship_xtra  is_hero  is_top_brand return_reason  discount  discount_rate  rating_average  review_count  favourite_count  has_ebook inventory_status inventory_type productset_group_name  data_version  day_ago_created  all_time_quantity_sold                                            authors                                     current_seller                                         categories                                     specifications                                        breadcrumbs  is_seller_in_chat_whitelist  is_tier_pricing_available  is_tier_pricing_eligible          crawl_time quantity_by_day
8      9     381234     381234  2436338861101  Cuốn Sách Hoàn Hảo Về Ngôn Ngữ Cơ Thể - Body L...        simple  Ngôn Ngữ Cơ Thể - Body Language (Tái Bản)Angel...  15900

In [6]:
books_df['authors'][0]

'[{"id": 5139, "name": "Minh Ni\\u1ec7m", "slug": "minh-niem"}]'

In [7]:
# Filter rows where quantity_by_day is not equal to zero
books_with_sales = books_df[books_df['quantity_by_day'] != 0]

# Display the result
books_with_sales


,id,tiki_id,master_id,sku,name,type,short_description,price,list_price,original_price,is_authentic,is_freeship_xtra,is_hero,is_top_brand,return_reason,discount,discount_rate,rating_average,review_count,favourite_count,has_ebook,inventory_status,inventory_type,productset_group_name,data_version,day_ago_created,all_time_quantity_sold,authors,current_seller,categories,specifications,breadcrumbs,is_seller_in_chat_whitelist,is_tier_pricing_available,is_tier_pricing_eligible,crawl_time,quantity_by_day
8,9,381234,381234,2436338861101,Cuốn Sách Hoàn Hảo Về Ngôn Ngữ Cơ Thể - Body L...,simple,Ngôn Ngữ Cơ Thể - Body Language (Tái Bản)Angel...,159000,198000,198000,1,0,1,0,any_reason,39000,20,4.8,1216,0,0,available,instock,,3300,0,32156,"[{""id"": 21427, ""name"": ""Allan & Barbara Pease""...","{""id"": 1, ""sku"": ""9786045806869"", ""name"": ""Tik...","{""id"": 871, ""name"": ""S\u00e1ch t\u01b0 duy - K...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-10-31 21:30:00,null
12,13,555504,555504,2434578841471,Sách Đắc Nhân Tâm (Bìa Cứng) (Tái Bản),simple,Đắc nhân tâm của Dale Carnegie là quyển sách d...,79200,108000,108000,1,0,1,0,any_reason,28800,27,4.8,1929,0,0,available,instock,,3300,2805,16014,"[{""id"": 133, ""name"": ""Dale Carnegie"", ""slug"": ...","{""id"": 1, ""sku"": ""8935086841204"", ""name"": ""Tik...","{""id"": 316, ""name"": ""S\u00e1ch ti\u1ebfng Vi\u...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-10-31 21:30:00,null
172,173,555504,555504,2434578841471,Sách Đắc Nhân Tâm (Bìa Cứng) (Tái Bản),simple,Đắc nhân tâm của Dale Carnegie là quyển sách d...,72000,108000,108000,1,0,1,0,any_reason,36000,33,4.8,1929,0,0,available,instock,,3300,2806,16017,"[{""id"": 133, ""name"": ""Dale Carnegie"", ""slug"": ...","{""id"": 1, ""sku"": ""8935086841204"", ""name"": ""Tik...","{""id"": 316, ""name"": ""S\u00e1ch ti\u1ebfng Vi\u...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-11-02 10:42:50,3.0
52,53,648672,648672,3105869967437,Đất Rừng Phương Nam (Tái Bản),simple,Tóm tắt nội dungCuộc sống lưu lạc của cậu bé A...,62400,81000,81000,1,0,1,0,any_reason,18600,23,4.8,206,0,0,available,instock,,3300,2722,2371,"[{""id"": 13152, ""name"": ""\u0110o\u00e0n Gi\u1ec...","{""id"": 1, ""sku"": ""3109233379079"", ""name"": ""Tik...","{""id"": 1754, ""name"": ""V\u0103n h\u1ecdc thi\u1...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-10-31 21:30:00,null
212,213,648672,648672,3105869967437,Đất Rừng Phương Nam (Tái Bản),simple,Tóm tắt nội dungCuộc sống lưu lạc của cậu bé A...,60800,81000,81000,1,0,1,0,any_reason,20200,25,4.8,206,0,0,available,instock,,3300,2723,2373,"[{""id"": 13152, ""name"": ""\u0110o\u00e0n Gi\u1ec...","{""id"": 1, ""sku"": ""3109233379079"", ""name"": ""Tik...","{""id"": 1754, ""name"": ""V\u0103n h\u1ecdc thi\u1...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-11-02 10:43:05,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32,33,275599939,275599939,5207908422455,Sưởi Ấm Mặt Trời: Tập 2 Của Cây Cam Ngọt,simple,"Sưởi Ấm Mặt Trời""Zezé, cậu bé tinh nghịch siêu...",124000,160000,160000,1,0,1,1,any_reason,36000,23,5.0,68,0,0,available,instock,,3300,112,1290,"[{""id"": 4292557, ""name"": ""Jos\u00e9 Mauro de V...","{""id"": 1, ""sku"": ""7957021308578"", ""name"": ""Tik...","{""id"": 6750, ""name"": ""Truy\u1ec7n d\u00e0i"", ""...","[{""name"": ""Th\u00f4ng tin chung"", ""attributes""...","[{""url"": ""/nha-sach-tiki/c8322"", ""name"": ""Nh\u...",1,0,0,2024-10-31 21:30:00,null
61,62,275702538,275702538,3188639955884,Chat GPT Thực

In [18]:
data = {
    'authors': [
        '[{"id": 5139, "name": "Minh Ni\\u1ec7m", "slug": "minh-niem"}]',
        '[{"id": 5140, "name": "Thich Nhat Hanh", "slug": "thich-nhat-hanh"}, {"id": 5141, "name": "Pema Chodron", "slug": "pema-chodron"}]'
    ]
}
authors_df = pd.DataFrame(data)
authors_df

,authors
0,"[{""id"": 5139, ""name"": ""Minh Ni\u1ec7m"", ""slug""..."
1,"[{""id"": 5140, ""name"": ""Thich Nhat Hanh"", ""slug..."


In [19]:
authors_df['authors'][0]

'[{"id": 5139, "name": "Minh Ni\\u1ec7m", "slug": "minh-niem"}]'

In [20]:
import json

authors_df['authors'] = authors_df['authors'].apply(json.loads)
authors_df

,authors
0,"[{'id': 5139, 'name': 'Minh Niệm', 'slug': 'mi..."
1,"[{'id': 5140, 'name': 'Thich Nhat Hanh', 'slug..."


In [24]:
authors_df['authors'][1]

[{'id': 5140, 'name': 'Thich Nhat Hanh', 'slug': 'thich-nhat-hanh'},
 {'id': 5141, 'name': 'Pema Chodron', 'slug': 'pema-chodron'}]

In [31]:
data = books_df['specifications'][0]
type(data)

str

In [32]:
data = json.loads(data)
type(data)

list

In [36]:
data[0]

{'name': 'Thông tin chung',
 'attributes': [{'code': 'book_version',
   'name': 'Phiên bản sách',
   'value': 'Phiên bản thường'},
  {'code': 'publisher_vn',
   'name': 'Công ty phát hành',
   'value': 'First News - Trí Việt'},
  {'code': 'publication_date',
   'name': 'Ngày xuất bản',
   'value': '2019-11-01 00:00:00'},
  {'code': 'dimensions', 'name': 'Kích thước', 'value': '13 x 20.5 cm'},
  {'code': 'book_cover', 'name': 'Loại bìa', 'value': 'Bìa mềm'},
  {'code': 'number_of_page', 'name': 'Số trang', 'value': '480'},
  {'code': 'manufacturer',
   'name': 'Nhà xuất bản',
   'value': 'Nhà Xuất Bản Tổng hợp TP.HCM'}]}

In [1]:
from transform import transform
from extract import query_data_from_sqliteDB

query = "SELECT * FROM books"
books_df = query_data_from_sqliteDB(query)
results_df = transform(books_df)
results_df

Successfully fetch data from sqliteDB!


,id,tiki_id,master_id,sku,name,type,short_description,price,list_price,original_price,...,dimensions,dich_gia,book_cover,number_of_page,manufacturer,seller_delivery_method,Organization_address,Organization_name,edition,bookcare_service
8,9,381234,381234,2436338861101,Cuốn Sách Hoàn Hảo Về Ngôn Ngữ Cơ Thể - Body L...,simple,Ngôn Ngữ Cơ Thể - Body Language (Tái Bản)Angel...,159000,198000,198000,...,14.5 x 20.5 cm,NaN,Bìa mềm,304,Nhà Xuất Bản Kim Đồng,NaN,NaN,NaN,NaN,NaN
88,89,381234,381234,2436338861101,Cuốn Sách Hoàn Hảo Về Ngôn Ngữ Cơ Thể - Body L...,simple,Ngôn Ngữ Cơ Thể - Body Language (Tái Bản)Angel...,159000,198000,198000,...,15.5 x 24 cm,NaN,Bìa mềm,530,Nhà Xuất Bản Hội Nhà Văn,Nhà bán giao hàng cho khách hàng,NaN,NaN,NaN,NaN
168,169,381234,381234,2436338861101,Cuốn Sách Hoàn Hảo Về Ngôn Ngữ Cơ Thể - Body L...,simple,Ngôn Ngữ Cơ Thể - Body Language (Tái Bản)Angel...,159000,198000,198000,...,13x20cm,NaN,Bìa mềm,348,NXB Trẻ,NaN,NaN,NaN,NaN,NaN
12,13,555504,555504,2434578841471,Sách Đắc Nhân Tâm (Bìa Cứng) (Tái Bản),simple,Đắc nhân tâm của Dale Carnegie là quyển sách d...,79200,108000,108000,...,<p>14 x 20.5 cm</p>,NaN,Bìa mềm,358,Nhà Xuất Bản Hội Nhà Văn,Nhà bán giao hàng cho khách hàng,NaN,NaN,NaN,NaN
92,93,555504,555504,2434578841471,Sách Đắc Nhân Tâm (Bìa Cứng) (Tái Bản),simple,Đắc nhân tâm của Dale Carnegie là quyển sách d...,70500,108000,108000,...,14 x 20.5 cm,NaN,Bìa mềm,428,Nhà Xuất Bản Hội Nhà Văn,Nhà bán giao hàng cho khách hàng,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,142,275702538,275702538,3188639955884,Chat GPT Thực Chiến,simple,"Trong thời đại hiện nay, tất cả những ai không...",105000,169000,169000,...,NaN,NaN,Bìa mềm,237,Nhà Xuất Bản Hồng Đức,NaN,NaN,NaN,2021,NaN
221,222,275702538,275702538,3188639955884,Chat GPT Thực Chiến,simple,"Trong thời đại hiện nay, tất cả những ai không...",113000,169000,169000,...,21 x 27 cm,1980 Books,Bìa cứng,128,Nhà Xuất Bản Dân Trí,NaN,NaN,NaN,NaN,NaN
19,20,276159943,276159943,4586901579152,NEXUS - Lược Sử Của Những Mạng Lưới Thông Tin ...,configurable,Giới thiệu sáchYuval Noah Harari trở lại với c...,254000,325000,325000,...,14 x 20.5cm,NaN,Bìa mềm,256,Nhà Xuất Bản Hội Nhà Văn,Nhà bán giao hàng cho khách hàng,NaN,NaN,NaN,NaN
99,100,276159943,276159943,4586901579152,NEXUS - Lược Sử Của Những Mạng Lưới Thông Tin ...,configurable,Giới thiệu sáchYuval Noah Harari trở lại với c...,254000,325000,325000,...,13 x 20.5 cm,Võ Hưng Thanh,Bìa gập,216,Nhà Xuất Bản Tổng hợp TP.HCM,Nhà bán giao hàng cho khách hàng,NaN,NaN,NaN,NaN
